In [1]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

import os
from enum import Enum

In [2]:
load_dotenv()

True

In [3]:
LLM_API_URL = os.environ["LLM_API_URL"]
LLM_API_TOKEN = os.environ["LLM_API_TOKEN"]
MODEL = "google/gemma-4-e2b"

In [4]:
# LLM_API_URL = os.environ["LMSTUDIO_BASE_URL"]
# LLM_API_TOKEN = os.environ["LM_API_TOKEN"]
# MODEL = "gemma-4-26B"

In [5]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)

# Modélisation du monde

In [6]:
VOID        = 0
PLAYER      = 1
ENNEMY      = 2
GOLD        = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

In [7]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [3, 1, 0, 0, 2, 0, 3], # (1, 1) # (1, 4) # (1, 6)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3], # (5, 6)
    [0, 0, 0, 0, 0, 0, 0],
])
initial_map

array([[0, 0, 0, 0, 0, 0, 0],
       [3, 1, 0, 0, 2, 0, 3],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3],
       [0, 0, 0, 0, 0, 0, 0]])

# Couche de contrat

In [8]:
H = "HAUT"
B = "BAS"
G = "GAUCHE"
D = "DROITE"

# H = "TOP"
# B = "DOWN"
# G = "LEFT"
# D = "RIGHT"

class Direction(str, Enum):
    HAUT       = H
    BAS        = B
    GAUCHE     = G
    DROITE     = D


class PlayerDecision(BaseModel):
    # directionJustification: str
    direction: Direction

MOVES = {
    H:   (-1,   0),
    B:   ( 1,   0),
    G:   ( 0,  -1),
    D:   ( 0,   1),
}

# Moteur de perception

In [9]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [10]:
def compute_distances(entities_positions, reference_pos):
    if (len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.abs(v).sum(axis=1)  # Manhattan (déplacements à 4 directions)
 
    return distances.astype(int)

In [11]:
def perception(world_map):

    player_position = localize(world_map, PLAYER)[0]
    golds_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    golds_distances = compute_distances(golds_positions, player_position)
    ennemies_distances = compute_distances(ennemies_positions, player_position)

    # Perception directionnelle : delta signé vers l'or le plus proche.
    nearest_gold_delta = {"row": 0, "col": 0}
    if len(golds_positions) > 0:
        nearest = golds_positions[int(np.argmin(golds_distances))]
        d_row, d_col = nearest - player_position
        nearest_gold_delta = {"row": int(d_row), "col": int(d_col)}

    return {
        "ennemies_distances": ennemies_distances.tolist(),
        "ennemies_count": len(ennemies_distances),
        "golds_distances": golds_distances.tolist(),
        "golds_count": len(golds_distances),
        "nearest_gold_delta": nearest_gold_delta,
    }

In [12]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('-----------------------------------------------------')

# Moteur de déplacement

In [13]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return False
    
    return world_map[r, c] in (VOID, GOLD)

In [14]:
def move(world_map: np.ndarray, old_pos, new_pos):
    move_result = {
        "gold_collected": False,
        "new_pos": old_pos
    }

    if not allowed_move(world_map, new_pos):
        return move_result

    entity = world_map[old_pos[0], old_pos[1]]
    target = world_map[new_pos[0], new_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    move_result["new_pos"] = new_pos

    if target == GOLD:
        move_result["gold_collected"] = True

    return move_result

# Moteur de décision

In [15]:
def decide(player_perception) -> PlayerDecision | None:
    prompt = f"""
    # Contexte
    - Tu es un joueur qui veut ramasser de l'or

    # Objectif
    - Trouve le plus court chemin vers l'or

    # Perception
    {player_perception}
    """

    # print(prompt)
    print(str(player_perception))

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=PlayerDecision,
        temperature=1
    )

    return response.choices[0].message.parsed or None

## Variante : décision via un modèle Cursor

Deux façons d'interroger un modèle Cursor (les deux renvoient du **texte** que
l'on parse manuellement vers `PlayerDecision`) :

- **`decide_cursor_cli`** — via le CLI `cursor-agent` (mode `ask`, lecture seule).
  Utilise la session `cursor-agent login`, **aucune `CURSOR_API_KEY` requise**.
- **`decide_cursor`** — via le SDK `cursor-sdk`. Nécessite une **vraie
  `CURSOR_API_KEY`** (Cursor Dashboard → Integrations) ; le login navigateur ne
  suffit pas pour le SDK.

On lance ensuite la boucle avec `game_loop(..., decide_fn=decide_cursor_cli)`.

In [16]:
import json
import re
import shutil
import subprocess
from cursor_sdk import Agent, AgentOptions, LocalAgentOptions, CursorAgentError

CURSOR_MODEL = "composer-2.5"


def cursor_pret() -> bool:
    """True si Cursor est utilisable : CURSOR_API_KEY réelle OU `cursor-agent` connecté."""
    cle = os.environ.get("CURSOR_API_KEY", "")
    if cle and "xxxx" not in cle:
        return True
    exe = shutil.which("cursor-agent")
    if not exe:
        return False
    try:
        res = subprocess.run([exe, "status"], capture_output=True, text=True, timeout=15)
        return "Not logged in" not in (res.stdout + res.stderr)
    except Exception:
        return False


def _extract_json(text: str) -> str | None:
    """Renvoie le dernier objet JSON *valide* trouvé dans le texte.

    La réponse peut contenir plusieurs accolades (ex. la perception ré-échoée) :
    on ne garde que le dernier bloc réellement parsable.
    """
    for bloc in reversed(re.findall(r"\{[^{}]*\}", text or "", re.S)):
        try:
            json.loads(bloc)
            return bloc
        except json.JSONDecodeError:
            continue
    return None


def decide_cursor(player_perception) -> PlayerDecision | None:
    """Variante de `decide` s'appuyant sur un modèle Cursor via le Cursor SDK."""
    prompt = f"""# Contexte
Tu es un joueur qui veut ramasser de l'or sur une grille.

# Objectif
Choisis la direction du prochain déplacement vers l'or le plus proche.

# Perception
{player_perception}

# Réponse attendue
Réponds UNIQUEMENT par un objet JSON, sans aucun texte autour :
{{"direction": "HAUT" | "BAS" | "GAUCHE" | "DROITE"}}"""

    print(str(player_perception))

    try:
        result = Agent.prompt(
            prompt,
            AgentOptions(
                # None -> le SDK utilise CURSOR_API_KEY (env) ou la session `cursor-agent login`.
                api_key=os.environ.get("CURSOR_API_KEY") or None,
                model=CURSOR_MODEL,
                local=LocalAgentOptions(cwd=os.getcwd()),
            ),
        )
    except Exception as e:  # CursorAgentError (auth/config/réseau) ou autre
        print(f"\t → Cursor SDK indisponible: {type(e).__name__}: {e}")
        return None

    if result.status != "finished":
        print(f"\t → run Cursor non terminé (status={result.status})")
        return None

    bloc = _extract_json(result.result)
    if bloc is None:
        print(f"\t → JSON introuvable dans la réponse: {result.result!r}")
        return None

    try:
        return PlayerDecision.model_validate_json(bloc)
    except Exception as e:
        print(f"\t → JSON non conforme à PlayerDecision: {e}")
        return None


def decide_cursor_cli(player_perception, model: str = "sonnet-4") -> PlayerDecision | None:
    """Variante Cursor via le CLI `cursor-agent` (session `cursor-agent login`).

    Contrairement au SDK, n'exige AUCUNE CURSOR_API_KEY : réutilise la connexion
    du CLI. Mode `ask` = lecture seule (aucune modification de fichier).
    """
    exe = shutil.which("cursor-agent")
    if not exe:
        print("\t → cursor-agent introuvable dans le PATH.")
        return None

    prompt = (
        "Tu joues sur une grille et veux ramasser l'or le plus proche.\n"
        f"Perception: {player_perception}\n"
        'Réponds UNIQUEMENT par un JSON {"direction": "HAUT"|"BAS"|"GAUCHE"|"DROITE"}.'
    )

    # Le CLI privilégie CURSOR_API_KEY : on retire une clé factice/absente pour
    # forcer l'usage de la session `cursor-agent login`.
    env = os.environ.copy()
    cle = env.get("CURSOR_API_KEY", "")
    if not cle or "xxxx" in cle:
        env.pop("CURSOR_API_KEY", None)

    print(str(player_perception))
    try:
        res = subprocess.run(
            [exe, "-p", "--mode", "ask", "--trust", "--model", model, prompt],
            capture_output=True, text=True, timeout=120, env=env,
        )
    except Exception as e:
        print(f"\t → cursor-agent indisponible: {type(e).__name__}: {e}")
        return None

    bloc = _extract_json(res.stdout)
    if bloc is None:
        print(f"\t → JSON introuvable (rc={res.returncode}) "
              f"stdout={res.stdout!r} stderr={res.stderr[:300]!r}")
        return None

    try:
        return PlayerDecision.model_validate_json(bloc)
    except Exception as e:
        print(f"\t → JSON non conforme à PlayerDecision: {e}")
        return None

# Game loop (simulation)

In [17]:
def game_loop(world_map: np.ndarray, max_turns = 10, decide_fn = decide, verbose = True):
    world_map = world_map.copy()
    move_history = []
    gold_collected = False
    turns_played = 0
    
    for turn in range(max_turns):
        turns_played = turn + 1
        if verbose:
            print(f"\n =================== [Turn {turn + 1}] ===================")
            show_map(world_map)

        player_pos = localize(world_map, PLAYER)[0]
        
        p = perception(world_map)
        p["move_history"] = move_history

        decision: PlayerDecision | None = decide_fn(p)
        if decision is None:
            continue

        if verbose:
            print(f"\t → decision: {decision.direction.value}")
        move_history.append(decision.direction.value)

        d_row, d_col = MOVES[decision.direction.value]
        new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)

        move_result = move(world_map, player_pos, new_pos)
        if move_result["gold_collected"]:
            gold_collected = True
            if verbose:
                print("FOUND GOLD !!!")
            break

    return {
        "success": gold_collected,
        "turns": turns_played,
        "moves": move_history,
        "map": world_map,
    }


In [18]:
# [RUN] Exécution avec le LLM OpenAI (uniquement si .env contient de vraies valeurs).
if LLM_API_URL and LLM_API_TOKEN and "xxxx" not in LLM_API_URL and "xxxx" not in LLM_API_TOKEN:
    game_loop(world_map=initial_map, max_turns=10, decide_fn=decide)
else:
    print("OpenAI non configuré (.env avec valeurs factices) -> cellule ignorée.")

OpenAI non configuré (.env avec valeurs factices) -> cellule ignorée.


In [19]:
# [RUN] Exécution avec un modèle Cursor.
# - decide_cursor_cli : via le CLI `cursor-agent login` (fonctionne SANS clé API).
# - decide_cursor      : via le SDK, nécessite une vraie CURSOR_API_KEY (Dashboard -> Integrations).
if cursor_pret():
    game_loop(world_map=initial_map, max_turns=10, decide_fn=decide_cursor_cli)
else:
    print("Cursor non authentifié -> `cursor-agent login` OU CURSOR_API_KEY dans .env, puis relance.")


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
💰	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
{'ennemies_distances': [3], 'ennemies_count': 1, 'golds_distances': [1, 5, 9], 'golds_count': 3, 'nearest_gold_delta': {'row': 0, 'col': -1}, 'move_history': []}


	 → decision: GAUCHE
FOUND GOLD !!!


# Cerveau de référence (hors-ligne)

`decide_greedy` est un cerveau **déterministe** qui suit `nearest_gold_delta`
vers l'or le plus proche — **sans aucun appel LLM**. Il sert de **baseline**
pour juger le LLM et permet d'exécuter le notebook de bout en bout avec le seul
kernel `.venv` (aucun identifiant requis).

In [20]:
def decide_greedy(player_perception) -> PlayerDecision | None:
    """Cerveau déterministe de référence (sans LLM).

    Suit `nearest_gold_delta` vers l'or le plus proche. Sert de baseline pour
    comparer le LLM et permet de faire tourner le notebook hors-ligne.
    """
    delta = player_perception.get("nearest_gold_delta", {"row": 0, "col": 0})
    d_row, d_col = delta["row"], delta["col"]

    if d_row == 0 and d_col == 0:
        return None

    # On progresse d'abord sur l'axe où l'écart est le plus grand.
    if abs(d_row) >= abs(d_col):
        direction = Direction.BAS if d_row > 0 else Direction.HAUT
    else:
        direction = Direction.DROITE if d_col > 0 else Direction.GAUCHE

    return PlayerDecision(direction=direction)

In [21]:
# [RUN] Démo hors-ligne : aucun LLM requis, tourne avec le seul kernel .venv.
resultat = game_loop(initial_map, max_turns=10, decide_fn=decide_greedy)
resultat


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
💰	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 → decision: GAUCHE
FOUND GOLD !!!


{'success': True,
 'turns': 1,
 'moves': ['GAUCHE'],
 'map': array([[0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 2, 0, 3],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 3],
        [0, 0, 0, 0, 0, 0, 0]])}

# Suivi en direct + métriques de performance

`game_loop_suivi` rejoue une partie en affichant un **tableau de bord live**
(grille, décision, latence, statut) et renvoie une **télémétrie détaillée**
(latence par tour, coups joués, blocages, décisions invalides).

`comparer` lance plusieurs parties sur des **cartes aléatoires** afin de
**traquer les performances** de chaque cerveau (taux de réussite, tours moyens,
latence par décision), et `graphe_comparaison` en produit un graphique.

In [ ]:
import time
from contextlib import nullcontext
from rich.console import Console, Group
from rich.live import Live
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

console = Console()


def rendre_grille(world_map) -> Table:
    """Grille en cases de largeur FIXE : la mise en page ne bouge pas d'un tour à
    l'autre, donc `rich` redessine en place sans scintillement."""
    grille = Table.grid(padding=(0, 1))
    for _ in range(world_map.shape[1]):
        grille.add_column(justify="center", width=2)
    for row in world_map:
        grille.add_row(*[Text(SYMBOLS.get(int(c), "?")) for c in row])
    return grille


def _panneau(world_map, tour, max_turns, direction, latence_ms, coups, statut):
    largeur = max(40, world_map.shape[1] * 3 + 8)  # largeur figée = pas de « saut » visuel
    stats = Table.grid(padding=(0, 2))
    stats.add_column(justify="right", style="bold", width=12)
    stats.add_column(width=largeur - 18)
    stats.add_row("Tour", f"{tour}/{max_turns}")
    stats.add_row("Décision", str(direction))
    stats.add_row("Latence", f"{latence_ms:.0f} ms")
    stats.add_row("Coups", str(len(coups)))
    stats.add_row("Statut", statut)
    return Group(
        Panel(rendre_grille(world_map), title="Carte", border_style="cyan", width=largeur),
        Panel(stats, title="Suivi", border_style="magenta", width=largeur),
    )


def game_loop_suivi(world_map, decide_fn=decide, max_turns=10, live=True, pause=0.0,
                    objectif="premier"):
    """game_loop instrumenté : tableau de bord live + télémétrie par tour.

    objectif : "premier" -> la partie s'arrête au 1er or ramassé (défaut) ;
               "tout"    -> la partie continue jusqu'à ramasser TOUT l'or
                            (ou d'atteindre max_turns).

    Renvoie un dict enrichi : success, turns, moves, ors_ramasses, invalides,
    duree_ms, latence_moyenne_ms, historique (une entrée par tour), map.
    """
    world_map = world_map.copy()
    coups, historique = [], []
    succes = False
    invalides = 0
    ors = 0
    t0 = time.perf_counter()

    ctx = (
        Live(_panneau(world_map, 0, max_turns, "—", 0, coups, "démarrage"),
             console=console, auto_refresh=False, vertical_overflow="crop")
        if live else nullcontext()
    )
    with ctx as vue:
        for turn in range(1, max_turns + 1):
            player_pos = localize(world_map, PLAYER)[0]
            avant = (int(player_pos[0]), int(player_pos[1]))

            p = perception(world_map)
            p["move_history"] = coups

            t = time.perf_counter()
            decision = decide_fn(p)
            latence = (time.perf_counter() - t) * 1000

            if decision is None:
                invalides += 1
                statut, direction = "décision invalide", "—"
                historique.append({"tour": turn, "direction": None,
                                   "latence_ms": round(latence), "bouge": False})
            else:
                direction = decision.direction.value
                d_row, d_col = MOVES[direction]
                res = move(world_map, player_pos, (avant[0] + d_row, avant[1] + d_col))
                apres = (int(res["new_pos"][0]), int(res["new_pos"][1]))
                bouge = apres != avant
                coups.append(direction)
                if res["gold_collected"]:
                    ors += 1
                    restant = int((world_map == GOLD).sum())
                    if objectif == "tout" and restant > 0:
                        statut = f"or {ors} ✓ (reste {restant})"
                    elif objectif == "tout":
                        succes, statut = True, f"TOUT L'OR 💰 ({ors})"
                    else:
                        succes, statut = True, "OR RAMASSÉ 💰"
                else:
                    statut = "en cours" if bouge else "bloqué"
                historique.append({"tour": turn, "direction": direction,
                                   "latence_ms": round(latence), "bouge": bouge})

            if live:
                vue.update(
                    _panneau(world_map, turn, max_turns, direction, latence, coups, statut),
                    refresh=True,
                )
                if pause:
                    time.sleep(pause)
            if succes:
                break

    duree = (time.perf_counter() - t0) * 1000
    latences = [h["latence_ms"] for h in historique]
    return {
        "success": succes,
        "turns": len(historique),
        "moves": coups,
        "ors_ramasses": ors,
        "invalides": invalides,
        "duree_ms": round(duree),
        "latence_moyenne_ms": round(sum(latences) / len(latences)) if latences else 0,
        "historique": historique,
        "map": world_map,
    }

In [ ]:
# [RUN] Suivi en direct — cerveau greedy (instantané)
game_loop_suivi(initial_map, decide_fn=decide_greedy, pause=0.6)

In [ ]:
import matplotlib.pyplot as plt


def carte_aleatoire(taille=7, n_or=3, n_ennemis=1, rng=None):
    """Carte aléatoire : 1 joueur, `n_or` ors et `n_ennemis` ennemis placés sans collision."""
    rng = rng or np.random.default_rng()
    grille = np.zeros((taille, taille), dtype=int)
    cases = [(r, c) for r in range(taille) for c in range(taille)]
    idx = rng.choice(len(cases), size=1 + n_or + n_ennemis, replace=False)
    choisies = [cases[i] for i in idx]
    (pr, pc), reste = choisies[0], choisies[1:]
    grille[pr, pc] = PLAYER
    for (r, c) in reste[:n_or]:
        grille[r, c] = GOLD
    for (r, c) in reste[n_or:]:
        grille[r, c] = ENNEMY
    return grille


def comparer(cerveaux: dict, n_cartes=10, max_turns=20, seed=0):
    """Rejoue les MÊMES cartes aléatoires pour chaque cerveau et agrège les métriques."""
    rng = np.random.default_rng(seed)
    cartes = [carte_aleatoire(rng=rng) for _ in range(n_cartes)]
    lignes = []
    for nom, fn in cerveaux.items():
        succes, tours, latences = 0, [], []
        for carte in cartes:
            r = game_loop_suivi(carte, decide_fn=fn, max_turns=max_turns, live=False)
            succes += int(r["success"])
            tours.append(r["turns"])
            latences.append(r["latence_moyenne_ms"])
        lignes.append({
            "cerveau": nom,
            "cartes": n_cartes,
            "taux_reussite": succes / n_cartes,
            "tours_moyens": sum(tours) / len(tours),
            "latence_moyenne_ms": sum(latences) / len(latences),
        })
    return lignes


def graphe_comparaison(lignes, chemin=None):
    """Bar charts : taux de réussite, tours moyens et latence moyenne par cerveau."""
    noms = [l["cerveau"] for l in lignes]
    couleurs = ["#4c9f70", "#c0504d", "#4f81bd", "#f2a900"][:len(noms)]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, cle, titre in zip(
        axes,
        ["taux_reussite", "tours_moyens", "latence_moyenne_ms"],
        ["Taux de réussite", "Tours moyens", "Latence moyenne / décision (ms)"],
    ):
        barres = ax.bar(noms, [l[cle] for l in lignes], color=couleurs)
        ax.set_title(titre)
        ax.bar_label(barres, fmt="%.2f")
    axes[0].set_ylim(0, 1.08)
    fig.tight_layout()
    if chemin:
        fig.savefig(chemin, dpi=110, bbox_inches="tight")
    return fig

In [ ]:
# [RUN] Benchmark de performance
LANCER_BENCH_CURSOR = False  # True = benchmark aussi le modèle Cursor (lent : appels réseau)

cerveaux = {"greedy": decide_greedy}
if LANCER_BENCH_CURSOR and cursor_pret():
    cerveaux["cursor"] = decide_cursor_cli

resultats = comparer(cerveaux, n_cartes=15, max_turns=25)
for r in resultats:
    print(r)
graphe_comparaison(resultats)

# ToDo 01/07


# Charge cognitive algorithmique VS LLM


## Stabilisation de la simulation

- Mettre en place la perception directionnelle

## Data ingénierie

Une fois le comportement de la simulation stabilisé :

- Choisir une structure de données propre pour synthétiser les données de la simulation
- Mettre en place un pipeline de data ingénierie pour processus les données de la simulation :
  - Architecture en médaillon (couche bronze → résultats bruts de la simulation, silver → données propres, gold → aggrégats métiers)
  - Utiliser ces données pour produire une datavisalisation (reporting) propre pour chaque "typologie" de simulation